# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c25_structural_check  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-29

## Goal
This notebook evaluates whether reclaimed stock members can safely carry predicted axial forces from the surrogate model, using Eurocode 5 based tension and buckling checks.

## Structural Check Logic
1. Predicted axial forces per edge are taken as demand values and multiplied by a safety margin (gnn_marge, default 1.10).
2. Characteristic timber strengths are converted to design strengths with $k_{mod}=0.8$ and $\gamma_M=1.3$.
3. For tension ($N \ge 0$), utilization is computed as demand over tensile resistance.
4. For compression ($N < 0$), utilization includes a buckling reduction factor $k_c$ derived from relative slenderness.
5. For every edge-stock combination, a utilization ratio is computed: values $\le 1.0$ are feasible, values $>1.0$ are unsafe.
6. Results are exported as long and matrix formats to support cost matrix generation and assignment optimization.

## Inputs
- Force table with edge ID, member length, and axial force (kN).
- Timber inventory with geometry and material properties: Width, Depth, Length, $f_{c,0,k}$, $f_{t,k}$, and $E_{0,mean}$.

## Outputs
- Utilization table (long format): one row per edge-stock combination.
- Utilization matrix (wide format): quick feasibility lookup.
- Safe options subset (only valid combinations with utilization $\le 1.0$).
- Slot table for downstream optimization and matching steps.

# IMPORT

In [3]:
import pandas as pd

import config
from src.structural_check import compute_utilization_outputs
from src.surrogate_io import (
    load_and_prepare_stock,
    load_surrogate_bundle,
    predict_edge_forces_kn,
)

# SURROGATE MODEL CHECK

In [4]:
# 1) Load trained surrogate model + scalers
bundle = load_surrogate_bundle(prefix_sm=None)  # Reads prefix from prefix_sm.txt by default

# 2) Select one design geometry row (13 nodes x 3 coordinates)
DESIGN_CSV = config.GH_DATA_PATH / "data_3.csv"
DESIGN_IDX = 0

df_designs = pd.read_csv(DESIGN_CSV)
design_row = df_designs.iloc[DESIGN_IDX]

# 3) Predict edge axial forces in kN for this geometry
#    Output columns: edge_id, V1, V2, length_m, axial_force_kn
df_forces = predict_edge_forces_kn(design_row, bundle)
print("Predicted force table preview:")
display(df_forces.head())

# 4) Load timber inventory and run Eurocode utilization checks
STOCK_CSV = config.TIMBER_STOCK_PATH / "complete_timber_small.csv"
gnn_marge = 1.10

df_input_stock = load_and_prepare_stock(STOCK_CSV)
outputs = compute_utilization_outputs(
    df_forces=df_forces[["edge_id", "length_m", "axial_force_kn"]],
    df_input_stock=df_input_stock,
    gnn_marge=gnn_marge,
)

# 5) Extract outputs used downstream in optimizer workflows
df_utilization_long = outputs["df_utilization_long"]
df_utilization_matrix = outputs["df_utilization_matrix"]
df_utilization_matrix_display = outputs["df_utilization_matrix_display"]
veilige_opties = outputs["veilige_opties"]
df_slots = outputs["df_slots"]

print("\nSafe options preview (utilization <= 1.0):")
display(veilige_opties.head(20))
print(f"Total safe combinations: {len(veilige_opties)}")

# Optional: write outputs for c26/c27 style downstream use
df_utilization_long.to_csv(config.EXPORT_PATH / "utilization_long_from_c25.csv", index=False)
df_utilization_matrix_display.to_csv(config.EXPORT_PATH / "utilization_matrix_from_c25.csv")
df_slots.to_csv(config.EXPORT_PATH / "slots_from_c25.csv", index=False)
print("\nExports written to 60_Research_Exports.")

Loaded surrogate prefix: data_3_0000
Device: cpu
Model: truss_edge_gnn_data_3_0000.pt
Predicted force table preview:


,edge_id,V1,V2,length_m,axial_force_kn
0,e0,0,1,4.125000,4.843127
1,e1,0,3,1.875000,4.835696
2,e2,1,2,1.875000,-9.692524
3,e3,1,4,3.557562,-2.155385
4,e4,2,5,3.375000,-7.070953



Safe options preview (utilization <= 1.0):


,Member_ID,Length,Width,Depth,utilization,edge_id,axial_force_kn,length_m
10,NS_00204,3300.0,38.0,100.0,0.162727,e0,4.843127,4.125
0,NS_00069,2100.0,50.0,100.0,0.123673,e0,4.843127,4.125
14,RS_00009,3000.0,50.0,150.0,0.104934,e0,4.843127,4.125
15,RS_00001,2850.0,50.0,150.0,0.104934,e0,4.843127,4.125
18,RS_00017,2950.0,50.0,150.0,0.104934,e0,4.843127,4.125
19,RS_00013,3100.0,50.0,150.0,0.104934,e0,4.843127,4.125
9,NS_00245,3600.0,63.0,150.0,0.065435,e0,4.843127,4.125
6,NS_00124,2400.0,50.0,225.0,0.054966,e0,4.843127,4.125
17,RS_00015,3600.0,70.0,230.0,0.048882,e0,4.843127,4.125
16,RS_00016,4050.0,80.0,210.0,0.046846,e0,4.843127,4.125


Total safe combinations: 586

Exports written to 60_Research_Exports.
